# RF-DETR training and test evaluation

This notebook trains RF-DETR Nano or Medium directly on the YOLO dataset in `dataset/`. The project’s RF-DETR 1.8.3 detects this format. It writes metrics and weights under `runs/`.

## Dependencies and GPU

Run this notebook in a Python 3.10+ environment with a CUDA GPU when possible.

In [ ]:
from pathlib import Path
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "dataset" / "data.yaml").is_file() and (p / "models").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the project repository.")
import torch
import yaml
print("Dependencies: install app/requirements.txt and notebooks/requirements.txt in the kernel environment.")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## Project dataset

Classes and train, validation and test folders are read from `dataset/`.

In [ ]:
DATASET_DIR = PROJECT_ROOT / "dataset"
with (DATASET_DIR / "data.yaml").open() as file:
    dataset_config = yaml.safe_load(file)
CLASS_NAMES = dataset_config["names"]
if isinstance(CLASS_NAMES, dict):
    CLASS_NAMES = [CLASS_NAMES[i] for i in sorted(CLASS_NAMES)]
required = [DATASET_DIR / split / subdir for split in ("train", "valid", "test") for subdir in ("images", "labels")]
missing = [path for path in required if not path.is_dir()]
if missing:
    raise FileNotFoundError("Missing project dataset folders: " + ", ".join(map(str, missing)))
print(f"Using project YOLO dataset: {DATASET_DIR}")
print(f"Classes: {CLASS_NAMES}")

## Configure and train

Choose `nano` or `medium`. The number of classes comes from `dataset/data.yaml`, including RF-DETR’s background class.

In [ ]:
from rfdetr import RFDETRMedium, RFDETRNano

VARIANT = "medium"  # "nano" or "medium"
NUM_CLASSES = len(CLASS_NAMES) + 1  # RF-DETR reserves class 0 for background.
EPOCHS = 50
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 1e-4
RESOLUTION = {"nano": 384, "medium": 576}[VARIANT]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = PROJECT_ROOT / "runs" / f"rfdetr-{VARIANT}-urban-disaster"
RUN_TEST = (DATASET_DIR / "test" / "images").is_dir()

model_class = {"nano": RFDETRNano, "medium": RFDETRMedium}[VARIANT]
model = model_class(num_classes=NUM_CLASSES)
model.train(
    dataset_dir=str(DATASET_DIR),
    output_dir=str(OUTPUT_DIR),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    lr=LEARNING_RATE,
    resolution=RESOLUTION,
    device=DEVICE,
    eval_interval=1,
    run_test=RUN_TEST,
)

## Inspect recorded metrics

RF-DETR writes training configuration and metrics to the run directory. Keep these files with the checkpoint so results can be compared with the YOLOv26m test evaluation.

In [ ]:
import pandas as pd

metrics_path = OUTPUT_DIR / "metrics.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    metric_columns = [
        column for column in ("epoch", "val/mAP_50", "val/mAP_50_95", "val/precision", "val/recall", "val/F1")
        if column in metrics.columns
    ]
    display(metrics[metric_columns].dropna(how="all").tail())
else:
    print(f"Run training first; metrics not found at {metrics_path}")

for artifact in ("training_config.json", "results.json", "checkpoint_best_total.pth"):
    print(f"{artifact}: {(OUTPUT_DIR / artifact).exists()}")

## Run inference with the best checkpoint

Use an image from the test split for a qualitative check. The trained checkpoint can also be uploaded to the project's Hugging Face repository for use by the Gradio app.

In [ ]:
image_extensions = {".jpg", ".jpeg", ".png", ".webp"}
test_images = sorted(path for path in (DATASET_DIR / "test" / "images").iterdir()
                     if path.suffix.lower() in image_extensions)
if not test_images:
    raise FileNotFoundError("No images found in dataset/test/images.")
checkpoint_path = OUTPUT_DIR / "checkpoint_best_total.pth"
if not checkpoint_path.is_file():
    raise FileNotFoundError(f"Best checkpoint not found: {checkpoint_path}")
best_model = model_class(num_classes=NUM_CLASSES, pretrain_weights=str(checkpoint_path))
detections = best_model.predict(str(test_images[0]), threshold=0.30)
print(f"Test image: {test_images[0].name}")
print(f"Detections: {len(detections.xyxy)}")
detections